In [ ]:
import pandas as pd
import numpy as np
import json
import geopandas as gpd
import h3
import os

In [ ]:
wac = pd.read_csv("../Data/ny_wac_S000_JT00_2023.csv.gz").rename(columns = {'w_geocode':'geocode'})
rac = pd.read_csv("../Data/ny_rac_S000_JT00_2023.csv.gz").rename(columns = {'h_geocode':'geocode'})
loc = pd.read_csv("../Data/ny_xwalk.csv.gz")

census = pd.read_csv("../Data/nhgis0001_csv/nhgis0001_ds258_2020_block.csv")

with open('../Output/Parcels/nyc_h3_parcels.geojson', "r") as f:
    hexagons = json.load(f)
hexagons = gpd.GeoDataFrame.from_features(hexagons["features"], crs="EPSG:4326")

blocks = gpd.read_file("../Data/census_block_geography/tl_2020_36_tabblock20.shp")

In [ ]:
loc_clean = loc[['tabblk2020','blklatdd', 'blklondd']].rename(columns = {'tabblk2020':'geocode', 'blklatdd':'latitude', 'blklondd':'longitude'})

def clean_data(df):
    white_collar_9_5 = ["CNS09", "CNS10", "CNS12", "CNS13", "CNS20"]
    shift_work_24_hrs = ["CNS03", "CNS05", "CNS08", "CNS14", "CNS16"]
    shfit_work_leisure = ["CNS07", "CNS11", "CNS17", "CNS18", "CNS19"]
    consistent_early_hrs = ["CNS15", "CNS04", "CNS06", "CNS01", "CNS02"]

    df_clean = df.copy()
    df_clean['white_collar_jobs'] = df[white_collar_9_5].sum(axis = 1)
    df_clean['all_day_jobs'] = df[shift_work_24_hrs].sum(axis = 1)
    df_clean['entertainment_jobs'] = df[shfit_work_leisure].sum(axis = 1)
    df_clean['early_start_jobs'] = df[consistent_early_hrs].sum(axis = 1)

    df_clean['age_29_or_younger'] = df_clean['CA01']
    df_clean['pay_1250_or_less'] = df_clean['CE01']
    df_clean['pay_1251_to_3333'] = df_clean['CE02']
    df_clean['pay_over_3333'] = df_clean['CE03']
    df_clean['sex_female'] = df_clean['CS02']

    df_clean.rename(columns = {'C000':'job_ct'}, inplace = True)
    df_clean = df_clean[['geocode', 'job_ct', 'white_collar_jobs','all_day_jobs',
                        'entertainment_jobs', 'early_start_jobs',
                        'age_29_or_younger', 'pay_1250_or_less',
                        'pay_1251_to_3333', 'pay_over_3333', 'sex_female']]
    df_clean = df_clean.fillna(0)

    return df_clean

In [58]:
work_demographics = clean_data(wac)
residential_demographics = clean_data(rac)

In [ ]:
nyc_census = census[census['COUNTY'].isin(['New York County', 'Queens County', 'Kings County', 'Bronx County', 'Richmond County'])]
nyc_census['age_15_to_34'] = (nyc_census['U7S006'] + nyc_census['U7S007'] + nyc_census['U7S008'] +
                                nyc_census['U7S009'] + nyc_census['U7S010'] + nyc_census['U7S011'] +
                                nyc_census['U7S012'] + nyc_census['U7S030'] + nyc_census['U7S031'] +
                                nyc_census['U7S032'] + nyc_census['U7S033'] + nyc_census['U7S034'] +
                                nyc_census['U7S035'] + nyc_census['U7S036'])
nyc_census.rename(columns = {'GEOCODE':'geocode', 'U7S001':'total_pop', 'U7S026':'female_pop', 'U9V001':'housing_units'}, inplace = True)
nyc_census = nyc_census[['geocode', 'total_pop', 'female_pop', 'age_15_to_34', 'housing_units']]
nyc_census = nyc_census.fillna(0)

census_demographics = nyc_census

new code

In [72]:
blocks['geocode'] = blocks['GEOID20']
blocks = blocks.to_crs(hexagons.crs)

block_hex_xwalk = gpd.sjoin(
    blocks[['geocode', 'geometry']],
    hexagons[['h3_index', 'geometry']],
    how='inner',
    predicate='intersects',
)[['geocode', 'h3_index']]
block_hex_xwalk['geocode'] = block_hex_xwalk['geocode'].astype(int)

In [66]:
def hex_merge(df, xwalk, hex, share_cols, denom_col, other_cols=[]):
    df_clean = df.copy()
    merged = xwalk.merge(df_clean, on='geocode', how='inner')

    all_metric_cols = list(set(share_cols + [denom_col] + other_cols))
    agg_dict = {col: 'mean' for col in all_metric_cols}
    hex_agg = merged.groupby('h3_index').agg(agg_dict).reset_index()

    for col in share_cols:
        hex_agg[col] = np.where(
            hex_agg[denom_col] == 0, 0, hex_agg[col] / hex_agg[denom_col]
        )

    res = hex[['h3_index', 'geometry']].merge(
        hex_agg, on='h3_index', how='inner'
    )
    return gpd.GeoDataFrame(res, geometry='geometry', crs=hex.crs)

In [ ]:
wac_rac_share_cols = [
    "white_collar_jobs",
    "all_day_jobs",
    "entertainment_jobs",
    "early_start_jobs",
    "age_29_or_younger",
    "pay_1250_or_less",
    "pay_1251_to_3333",
    "pay_over_3333",
    "sex_female",
]

hex_work = hex_merge(work_demographics, block_hex_xwalk, hexagons, wac_rac_share_cols, "job_ct", [])
hex_residential = hex_merge(residential_demographics, block_hex_xwalk, hexagons, wac_rac_share_cols, "job_ct", [])

census_share_cols = [
        "female_pop",
        "age_15_to_34",]

hex_census = hex_merge(census_demographics, block_hex_xwalk, hexagons, census_share_cols, "total_pop", ['housing_units'])

hex_work.to_csv('../Intermediate/Census/work_employment_data.csv', index=False)
hex_residential.to_csv('../Intermediate/Census/work_residential_data.csv', index=False)
hex_census.to_csv('../Intermediate/Census/census_data.csv', index=False)

In [86]:
work_type_map = hex_work.explore(
    column="white_collar_jobs",
    cmap="PuBuGn",
    legend=True,
    tiles="CartoDB positron",
    tooltip=[
        "h3_index",
        "all_day_jobs",
        "entertainment_jobs",
        "early_start_jobs"
        ],
    popup=True,
)
work_type_map.save("../Output/Exploratory Maps/work_type_map.html")

work_pay_map = hex_work.explore(
    column="job_ct",
    cmap="PuBuGn",
    legend=True,
    tiles="CartoDB positron",
    tooltip=[
        "h3_index",
        "pay_1250_or_less",
        "pay_1251_to_3333",
        "pay_over_3333"
        ],
    popup=True,
)
work_pay_map.save("../Output/Exploratory Maps/work_pay_map.html")

res_type_map = hex_residential.explore(
    column="white_collar_jobs",
    cmap="PuBuGn",
    legend=True,
    tiles="CartoDB positron",
    tooltip=[
        "h3_index",
        "all_day_jobs",
        "entertainment_jobs",
        "early_start_jobs"
        ],
    popup=True,
)
res_type_map.save("../Output/Exploratory Maps/res_type_map.html")

res_pay_map = hex_residential.explore(
    column="job_ct",
    cmap="PuBuGn",
    legend=True,
    tiles="CartoDB positron",
    tooltip=[
        "h3_index",
        "pay_1250_or_less",
        "pay_1251_to_3333",
        "pay_over_3333"
        ],
    popup=True,
)
res_pay_map.save("../Output/Exploratory Maps/res_pay_map.html")

census_map = hex_census.explore(
    column="total_pop",
    cmap="PuBuGn",
    legend=True,
    tiles="CartoDB positron",
    tooltip=[
        "h3_index",
        "total_pop",
        "age_15_to_34",
        "female_pop",
        "housing_units",
    ],
    popup=True,
)
census_map.save("../Output/Exploratory Maps/census_housing_pop_map.html")